Downloads GTFS feeds from official URLs and extracts them to Volumes. \
Run before 01_ingest_bronze.py whenever feeds need to be refreshed.

# 1. GLOBAL CONFIG
Variables used by all the other notebooks. \
Edit here if path changes or new cities are added.

In [0]:
%run ./00_utils.py

In [0]:
cfg = load_config()

BASE_PATH = cfg["base_path"]
print(f"Base path: {BASE_PATH}")

CITIES = cfg["cities"]
print(f"Cities: {CITIES}")

GTFS_TABLES = cfg["gtfs_tables"]
print(f"GTFS tables: {GTFS_TABLES}")

CITIES_WITHOUT_CALENDAR = cfg["cities_without_calendar"]
print(f"Cities without calendar: {CITIES_WITHOUT_CALENDAR}")

In [0]:
import zipfile
import os
import shutil
import urllib.request

# Download and extract GTFS feed for each city

In [0]:
for city, config in CITIES.items():
    url = config["url"]
    folder = config["folder"]
    target_path = f"{BASE_PATH}/{folder}"
    tmp_zip = f"/tmp/gtfs_{city}.zip"

    print(f"\n=== {city} ===")

    # Create target folder if missing
    if not os.path.exists(target_path):
        os.makedirs(target_path, exist_ok=True)
        print(f"  ✓ created folder → {target_path}")

    # Download zip
    try:
        req = urllib.request.Request(url, 
                                     headers={"User-Agent": "Mozilla/5.0",
                                              "Accept": "application/zip, application/octet-stream, */*"})
        with urllib.request.urlopen(req) as r:
            with open(tmp_zip, "wb") as f:
                f.write(r.read())
        print(f"  ✓ downloaded → {tmp_zip}")
    except Exception as e:
        print(f"  ✗ download failed → {e}")
        continue

    # Extract txt files
    try:
        if is_gtfs:
            # file .gtfs è uno zip rinominato — prova a leggerlo come zip
            shutil.copy(tmp_zip, f"/tmp/gtfs_{city}/gtfs.zip")
            with zipfile.ZipFile(f"/tmp/gtfs_{city}/gtfs.zip", "r") as z:
                txt_files = [f for f in z.namelist() if f.endswith(".txt")]
                z.extractall(f"/tmp/gtfs_{city}/")
        else:
            with zipfile.ZipFile(tmp_zip, "r") as z:
                txt_files = [f for f in z.namelist() if f.endswith(".txt")]
                z.extractall(f"/tmp/gtfs_{city}/")
        print(f"  ✓ extracted {len(txt_files)} files")
    except Exception as e:
        print(f"  ✗ extraction failed → {e}")
        continue

    # Copy to Volumes
    for txt_file in txt_files:
        src = f"/tmp/gtfs_{city}/{txt_file}"
        dst = f"{target_path}/{txt_file}"
        if not os.path.exists(dst):
            with open(dst, "w") as f:
                pass
            print(f"  ✓ created empty file → {dst}")
        try:
            shutil.copy(src, dst)
        except Exception as e:
            print(f"  ✗ copy failed {txt_file} → {e}")

    print(f"  ✓ copied to {target_path}")